In [1]:
import networkx as nx
import os,pickle
import sys
from scipy import stats
sys.path.append('../include')
from load_dictionary import load_dictionary
import db_init as db
import pandas as pd

In [2]:
dicttimestamp = '20180706'
engine = db.connectToPostgreSQL(server='cns-postgres-myaura')
tablename = 'dictionaries.dict_%s' % (dicttimestamp)
sql = """
    SELECT
        d.id,
        COALESCE(d.id_parent,d.id) AS id_parent,
        d.dictionary,
        d.token,
        COALESCE(p.token, d.token) as parent,
        d.type,
        d.source,
        d.id_original,
        COALESCE(p.id_original, d.id_original) as id_original_parent
        FROM %s d
        LEFT JOIN %s p ON d.id_parent = p.id
        """ % (tablename, tablename)
dfD = pd.read_sql(sql, engine, index_col='id')

In [29]:
# old_network_file = '../instagram/tmp-data/04-instagram-epilepsy-network-20180706-samepost.graphml'
# new_network_file = '../instagram/tmp-data/04-instagram-epilepsy-network-20210321-samepost.graphml'
# old_network_file = '../efwebsite/tmp-data/04-efwebsite-forums-network-20180706-samepost.graphml'
# new_network_file = '../efwebsite/tmp-data/04-efwebsite-forums-network-20210321-samepost.graphml'
# old_network_file = '../pubmed/tmp-data/04-pubmed-epilepsy-network-20180706.graphml'
# new_network_file = '../pubmed/tmp-data/04-pubmed-epilepsy-network-20210321.graphml'
old_network_file = '../clinicaltrials/tmp-data/04-pubmed-epilepsy-network-20180706.graphml'
new_network_file = '../clinicaltrials/tmp-data/04-pubmed-epilepsy-network-20210321.graphml'
old_net = nx.read_graphml(old_network_file)
new_net = nx.read_graphml(new_network_file)

In [4]:
removed_id = [8433,202468,211750,201798,201791,35150,174899,240710,8619,199415,170326,190790]

def get_parent(idx):
    return dfD.loc[idx]['id_parent']

removed_parent = [get_parent(i) for i in removed_id]

In [30]:
# eigen_old = nx.eigenvector_centrality_numpy(old_net, weight='count')
# eigen_new = nx.eigenvector_centrality_numpy(new_net, weight='count')
eigen_old = nx.pagerank_numpy(old_net, alpha=0.99, weight='count')
eigen_new = nx.pagerank_numpy(new_net, alpha=0.99, weight='count')
# eigen_old = nx.eigenvector_centrality_numpy(old_net, weight='proximity')
# eigen_new = nx.eigenvector_centrality_numpy(new_net, weight='proximity')
# eigen_old = nx.pagerank_numpy(old_net, alpha=0.99, weight='proximity')
# eigen_new = nx.pagerank_numpy(new_net, alpha=0.99, weight='proximity')


In [7]:
def get_token(idx):
    return dfD.loc[int(idx)]['token']
def print_list(l):
    for line in l:
        print('|'+'|'.join([str(i) for i in line])+'|')
def describe_list(l, d):
    for item in l:
        token = get_token(item)
        print('|', item,'|',  token,'|',  d[item], '|' )

In [31]:
top_nodes_old = sorted(eigen_old, key=eigen_old.get, reverse=True)
describe_list(top_nodes_old[:20], eigen_old)

| 174366 | Convulsion | 0.04767065507189911 |
| 176107 | Epilepsy | 0.04621130488784895 |
| 175720 | Electroencephalogram | 0.01614613587384173 |
| 542 | Lamotrigine | 0.01254248270080677 |
| 175023 | Depression | 0.011758516351977647 |
| 186589 | Surgery | 0.01158926157899134 |
| 1186 | Levetiracetam | 0.011588972404360598 |
| 182927 | Partial seizures | 0.011094696495046378 |
| 551 | Carbamazepine | 0.011044022627839966 |
| 304 | Valproic Acid | 0.009821497644666258 |
| 182630 | Pain | 0.009399928224266631 |
| 169902 | Agitation | 0.00936191885541322 |
| 171894 | Blindness | 0.009139873314784277 |
| 170661 | Anxiety | 0.008017636090725276 |
| 815 | Diazepam | 0.00761340549609694 |
| 185679 | Sedation | 0.006688457499784331 |
| 181699 | Nervousness | 0.00663652586837419 |
| 184239 | Prophylaxis | 0.006564911211251072 |
| 243 | Phenytoin | 0.006447652212927554 |
| 179153 | Injury | 0.006026803819018969 |


In [32]:
top_nodes_new = sorted(eigen_new, key=eigen_new.get, reverse=True)
describe_list(top_nodes_new[:20], eigen_new)

| 174366 | Convulsion | 0.04804100525687079 |
| 176107 | Epilepsy | 0.04652060107571291 |
| 175720 | Electroencephalogram | 0.016241297770259205 |
| 542 | Lamotrigine | 0.012671717512029692 |
| 175023 | Depression | 0.011799496952198634 |
| 1186 | Levetiracetam | 0.011696718087882063 |
| 186589 | Surgery | 0.01169623960048735 |
| 182927 | Partial seizures | 0.011176594022197485 |
| 551 | Carbamazepine | 0.01117169143611138 |
| 304 | Valproic Acid | 0.00993278778221025 |
| 182630 | Pain | 0.009465943240469432 |
| 169902 | Agitation | 0.009447202908070529 |
| 171894 | Blindness | 0.00915241105862256 |
| 170661 | Anxiety | 0.008054348118373023 |
| 185679 | Sedation | 0.006725749846872692 |
| 181699 | Nervousness | 0.006691852468773271 |
| 815 | Diazepam | 0.006651668330826893 |
| 184239 | Prophylaxis | 0.006609360005461427 |
| 243 | Phenytoin | 0.006510333089588703 |
| 179153 | Injury | 0.006083352115595013 |


In [32]:
common_id = set(top_nodes_new) & set(top_nodes_old)
legit_id = common_id - set([str(i) for i in removed_parent])

limit = 500
count = 0
rg1 = []
idl = []
rg2 = []
for i, idd in enumerate(top_nodes_old):
    if idd in legit_id:
        count += 1
        if count > limit:
            break
        rg1.append(i)
        idl.append(idd)
for idd in idl:
    rg2.append(top_nodes_new.index(idd))
print(rg1)
print(rg2)

[1, 2, 3, 4, 5, 6, 7, 8, 10, 11, 12, 13, 14, 15, 17, 19, 20, 21, 22, 23, 24, 25, 26, 28, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 199, 200, 201, 202, 203, 204, 205, 206, 207, 208, 209, 210, 211, 212, 213, 214, 215, 216, 217, 218, 219, 220, 221, 222, 223, 224, 225, 226, 227, 228, 229

In [33]:
for i in [10, 20, 50, 100, 200, 500]:
    rho, p = stats.spearmanr(rg1[:i], rg2[:i])
    print('|', '|'.join([str(x) for x in [i,rho,p]] ), '|')

| 10|0.9515151515151514|2.279854920641689e-05 |
| 20|0.9834586466165413|8.293403515243347e-15 |
| 50|0.9805042016806722|1.4075473594545406e-35 |
| 100|0.9697209720972096|8.319219055751145e-62 |
| 200|0.9826280657016426|8.559450513960577e-147 |
| 500|0.9872436609746438|0.0 |


In [34]:
for i in [10, 20, 50, 100, 200, 500]:
    rho, p = stats.kendalltau(rg1[:i], rg2[:i])
    print('|', '|'.join([str(x) for x in [i,rho,p]] ), '|')

| 10|0.8666666666666666|0.00011518959435626102 |
| 20|0.9157894736842106|1.637252132068114e-12 |
| 50|0.9004081632653061|2.797809088806884e-20 |
| 100|0.873939393939394|5.586950316008961e-38 |
| 200|0.9033165829145728|1.843392328391386e-80 |
| 500|0.9342685370741483|4.583372334712145e-214 |


In [97]:
for term in top_nodes_new:
    print(term, get_token(term))
    prox_l = []
    for to_node in old_net[term]:
        prox_l.append([get_token(to_node), old_net[term][to_node]['proximity']])
    prox_l.sort(key=lambda x:x[1],reverse=True)
    print_list(prox_l[:5])
    print('--------------')
    prox_l = []
    for to_node in new_net[term]:
        prox_l.append([get_token(to_node), new_net[term][to_node]['proximity']])
    prox_l.sort(key=lambda x:x[1],reverse=True)
    print_list(prox_l[:5])

175023 Depression
|Anxiety|0.10830384453907516|
|Completed suicide|0.060419602627974835|
|Decreased appetite|0.03624554358070729|
|Insomnia|0.03427923251432307|
|Pain|0.026145774617352468|
--------------
|Anxiety|0.11741278975259274|
|Completed suicide|0.06601318528799445|
|Decreased appetite|0.03879811382827064|
|Insomnia|0.03707785726168691|
|Pain|0.02821404619679648|
170661 Anxiety
|Depression|0.10830384453907516|
|Insomnia|0.03419830028328612|
|Completed suicide|0.033331811332815854|
|Stress|0.022398345968297727|
|Decreased appetite|0.02197590361445783|
--------------
|Depression|0.11741278975259274|
|Insomnia|0.0369409287865064|
|Completed suicide|0.03633377298857726|
|Stress|0.024170480549199083|
|Decreased appetite|0.02342124861963584|
174010 Completed suicide
|Depression|0.060419602627974835|
|Death|0.04231893341874524|
|Decreased appetite|0.039351412857955625|
|Schizophrenia|0.03488817891373802|
|Bulimia nervosa|0.034357936069697524|
--------------
|Depression|0.06601318528799

In [98]:
for term in top_nodes_old:
    if term in new_net:
        print(term, get_token(term))
        prox_l = []
        for to_node in old_net[term]:
            prox_l.append([get_token(to_node), old_net[term][to_node]['proximity']])
        prox_l.sort(key=lambda x:x[1],reverse=True)
        print_list(prox_l[:5])
        print('--------------')
        prox_l = []
        for to_node in new_net[term]:
            prox_l.append([get_token(to_node), new_net[term][to_node]['proximity']])
        prox_l.sort(key=lambda x:x[1],reverse=True)
        print_list(prox_l[:5])

176626 Feeling hot
|Euphoric mood|0.41945874827030827|
|Cocoa|0.0344941783448372|
|Homosexuality|0.029198445520480475|
|Nasopharyngitis|0.026516596718809616|
|Tattoo|0.02079722703639515|
--------------
|Tanning|0.014373716632443531|
|Chills|0.008403361344537815|
|Feeling abnormal|0.007658643326039387|
|Tenderness|0.006382978723404255|
|Feeling of relaxation|0.005908419497784343|
176223 Euphoric mood
|Feeling hot|0.41945874827030827|
|Trance|0.004973692864189411|
|Cannabis|0.0045064599483204135|
|Mania|0.002979373567608862|
|Investigation|0.002690221046495987|
--------------
|Epinephrine|0.008982035928143712|
|Irritability|0.0071174377224199285|
|Diphenhydramine|0.006880733944954129|
|Hypomania|0.004076086956521739|
|Quetiapine|0.0023215322112594312|
8976 Cocoa
|Vanilla|0.042611768573307035|
|Feeling hot|0.0344941783448372|
|Coffee bean|0.03331500824628917|
|Strawberry|0.030600461893764433|
|Banana|0.029295197833410398|
--------------
|Vanilla|0.04972667114222691|
|Coffee bean|0.0399881

In [ ]:
node_list = []
for node in old_net:
    weight = 0.0
    for edge_to in old_net[node]:
        weight += old_net[node][edge_to]['proximity']
    node_list.append([node, weight])